# Loan Prediction Competition

In this notebook, we apply ensemble methods such as XGBoost for binary loan approval classification, featuring systematic hyperparameter optimization and SHAP model interpretability.


## 1. Setup & Configuration


In [ ]:
import os
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Configuration constants — edit DATA_DIR to point to a different directory
DATA_DIR = '.'
RANDOM_STATE = 42
CV_FOLDS = 5


**Dependencies:** pandas, numpy, scikit-learn, xgboost, shap, matplotlib, seaborn

Install with: `pip install -r requirements.txt`


## 2. Data Loading & Exploration

Train and test data are loaded once here and reused in later sections — no duplicate loads.


In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_no_label.csv'))
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()


In [ ]:
train_df.describe()


In [ ]:
# =====================================
# Distribution exploration
# =====================================
import matplotlib.pyplot as plt
import seaborn as sns

# Define variable lists
categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
               'Credit_History', 'Property_Area', 'Loan_Status']  # target included to inspect class balance
numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

# -------------------------
# Categorical variables: count plots
# -------------------------
for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=train_df, palette="pastel")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

# -------------------------
# Numerical variables: histogram + boxplot
# -------------------------
for col in numerical_cols:
    fig, axes = plt.subplots(1,2, figsize=(12,4))

    # Histogram
    sns.histplot(train_df[col].dropna(), kde=True, ax=axes[0], color="skyblue")
    axes[0].set_title(f"Histogram of {col}")

    # Boxplot to inspect outliers
    sns.boxplot(x=train_df[col], ax=axes[1], color="lightgreen")
    axes[1].set_title(f"Boxplot of {col}")

    plt.tight_layout()
    plt.show()

In [ ]:
# Missing value analysis
missing_values = train_df.isnull().sum()
missing_percent = (missing_values / len(train_df)) * 100

missing_summary = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percent
}).sort_values(by='Percentage (%)', ascending=False)

print("Missing values summary:")
missing_summary

## 3. Preprocessing & Feature Engineering

The preprocessing pipeline handles missing values via intelligent imputation, encodes categorical features, applies log transforms to reduce skew in income and loan amount columns, and engineers domain-specific risk features.


In [ ]:
def preprocess_loan_data(df):
    """
    Preprocessing pipeline for loan prediction data.

    Handles missing values via intelligent imputation, encodes categorical
    features, applies log transforms to reduce skew, and engineers
    domain-specific risk features.

    Args:
        df: Raw loan DataFrame (train or test set)

    Returns:
        Processed DataFrame ready for model training
    """

    # Crear copia para no modificar original
    data = df.copy()

    # ===================
    # 1. IMPUTACIÓN INTELIGENTE
    # ===================

    # ✅ CLAVE: Manejo inteligente de Credit_History (esto dio el salto en accuracy)
    data['Credit_History_Missing'] = data['Credit_History'].isna().astype(int)
    data['Credit_History'].fillna(1.0, inplace=True)

    # Imputaciones simples
    data['Self_Employed'].fillna('No', inplace=True)
    data['Loan_Amount_Term'].fillna(360.0, inplace=True)
    data['Gender'].fillna('Male', inplace=True)
    data['Dependents'].fillna('0', inplace=True)
    data['Married'].fillna('No', inplace=True)
    data['Education'].fillna('Graduate', inplace=True)
    data['CoapplicantIncome'].fillna(0.0, inplace=True)

    # Imputación por grupos para ApplicantIncome
    if data['ApplicantIncome'].isnull().sum() > 0:
        median_by_group = data.groupby(['Education', 'Self_Employed'])['ApplicantIncome'].median()
        for idx in data[data['ApplicantIncome'].isnull()].index:
            education = data.loc[idx, 'Education']
            self_employed = data.loc[idx, 'Self_Employed']
            if (education, self_employed) in median_by_group.index:
                data.loc[idx, 'ApplicantIncome'] = median_by_group[education, self_employed]
            else:
                data.loc[idx, 'ApplicantIncome'] = data['ApplicantIncome'].median()

    # Imputación por grupos para LoanAmount
    if data['LoanAmount'].isnull().sum() > 0:
        median_by_group = data.groupby(['Education', 'Property_Area'])['LoanAmount'].median()
        for idx in data[data['LoanAmount'].isnull()].index:
            education = data.loc[idx, 'Education']
            property_area = data.loc[idx, 'Property_Area']
            if (education, property_area) in median_by_group.index:
                data.loc[idx, 'LoanAmount'] = median_by_group[education, property_area]
            else:
                data.loc[idx, 'LoanAmount'] = data['LoanAmount'].median()

    # ===================
    # Log transforms reduce right skew in income and loan amount features.
    # Normalized distributions improve XGBoost split quality.
    # 2. TRANSFORMACIONES LOGARÍTMICAS
    # ===================

    data['log_ApplicantIncome'] = np.log(data['ApplicantIncome'])
    data['log_CoapplicantIncome'] = np.log1p(data['CoapplicantIncome'])
    data['log_LoanAmount'] = np.log(data['LoanAmount'])

    # ===================
    # 3. FEATURE ENGINEERING
    # ===================

    # Variables base
    total_income = data['ApplicantIncome'] + data['CoapplicantIncome']

    # Features principales
    data['total_income'] = total_income
    data['loan_to_income_ratio'] = data['LoanAmount'] / (total_income / 12)
    data['coapplicant_contribution'] = data['CoapplicantIncome'] / total_income

    # Income per person
    dependents_numeric = data['Dependents'].replace('3+', '3').astype(int) + 1
    data['income_per_person'] = total_income / dependents_numeric

    # Ratios logarítmicos
    log_monthly_payment = data['log_LoanAmount'] - np.log(data['Loan_Amount_Term'])
    log_monthly_income = np.log(total_income) - np.log(12)
    data['log_monthly_payment_to_income_ratio'] = log_monthly_payment - log_monthly_income

    # Categorización
    data['income_category'] = pd.cut(total_income,
                                   bins=[0, 3000, 6000, 10000, np.inf],
                                   labels=['Low', 'Medium', 'High', 'Very_High'])

    data['loan_size_category'] = pd.cut(data['LoanAmount'],
                                      bins=[0, 100, 150, 200, np.inf],
                                      labels=['Small', 'Medium', 'Large', 'Very_Large'])

    # Perfiles de riesgo y estabilidad
    # Engineered risk features capture domain knowledge about loan creditworthiness:
    # - high_risk_profile: applicant has multiple simultaneous risk factors
    # - financial_stability: stable married income, no outstanding debt
    data['high_risk_profile'] = ((data['Credit_History'] == 0) |
                                (data['Self_Employed'] == 'Yes') |
                                (data['loan_to_income_ratio'] > 0.4)).astype(int)

    married_flag = (data['Married'] == 'Yes').astype(int)
    education_flag = (data['Education'] == 'Graduate').astype(int)
    data['financial_stability'] = (married_flag &
                                  education_flag &
                                  (data['Credit_History'] == 1.0)).astype(int)

    # ✅ FEATURE INTERACTIONS (clave para el 80%)
    data['credit_missing_interaction'] = data['Credit_History'] * data['Credit_History_Missing']
    data['married_coapplicant_synergy'] = married_flag * (data['CoapplicantIncome'] > 0).astype(int)

    education_flag = (data['Education'] == 'Graduate').astype(int)
    self_employed_flag = (data['Self_Employed'] == 'Yes').astype(int)
    data['education_employment_interaction'] = education_flag * self_employed_flag

    # Wealthy conservative borrower
    income_q75 = total_income.quantile(0.75)
    loan_q25 = data['LoanAmount'].quantile(0.25)
    high_income_flag = (total_income > income_q75)
    small_loan_flag = (data['LoanAmount'] < loan_q25)
    data['wealthy_conservative_borrower'] = (high_income_flag & small_loan_flag).astype(int)

    # Term-amount mismatch
    long_term = (data['Loan_Amount_Term'] >= 360)
    short_term = (data['Loan_Amount_Term'] < 360)
    large_loan = (data['LoanAmount'] > data['LoanAmount'].quantile(0.75))
    small_loan = (data['LoanAmount'] <= data['LoanAmount'].quantile(0.25))
    data['term_amount_mismatch'] = ((long_term & small_loan) | (short_term & large_loan)).astype(int)

    # Property-income context
    urban_flag = (data['Property_Area'] == 'Urban')
    rural_flag = (data['Property_Area'] == 'Rural')
    high_income_percentile = (total_income > total_income.quantile(0.8))
    data['urban_high_income'] = (urban_flag & high_income_percentile).astype(int)
    data['rural_high_income'] = (rural_flag & high_income_percentile).astype(int)

    # ===================
    # 4. ENCODING
    # ===================

    # Variables binarias
    data['Gender_Male'] = (data['Gender'] == 'Male').astype(int)
    data['Married_Yes'] = (data['Married'] == 'Yes').astype(int)
    data['Self_Employed_Yes'] = (data['Self_Employed'] == 'Yes').astype(int)
    data['Education_Graduate'] = (data['Education'] == 'Graduate').astype(int)

    # One-Hot Encoding
    dependents_dummies = pd.get_dummies(data['Dependents'], prefix='Dependents', drop_first=True)
    property_dummies = pd.get_dummies(data['Property_Area'], prefix='Property_Area', drop_first=True)
    income_dummies = pd.get_dummies(data['income_category'], prefix='Income', drop_first=True)
    loan_size_dummies = pd.get_dummies(data['loan_size_category'], prefix='LoanSize', drop_first=True)

    # Concatenar dummies
    data = pd.concat([data, dependents_dummies, property_dummies,
                     income_dummies, loan_size_dummies], axis=1)

    # Target encoding
    if 'Loan_Status' in data.columns:
        data['Loan_Status'] = (data['Loan_Status'] == 'Y').astype(int)

    # ===================
    # 5. LIMPIEZA FINAL
    # ===================

    # Eliminar variables categóricas originales
    categorical_to_drop = ['Gender', 'Married', 'Self_Employed', 'Education',
                          'Dependents', 'Property_Area', 'income_category', 'loan_size_category']

    # Eliminar transformaciones menos útiles
    numerical_to_drop = ['log_CoapplicantIncome']

    # Eliminar solo las que existen
    all_to_drop = [col for col in categorical_to_drop + numerical_to_drop if col in data.columns]
    data.drop(all_to_drop, axis=1, inplace=True)

    return data

In [ ]:
# Illustrative: show feature count before and after preprocessing.
# The actual training pipeline in Section 4 re-applies this to train_df.
_processed = preprocess_loan_data(train_df.copy())
print(f'Features before preprocessing: {train_df.shape[1]}')
print(f'Features after preprocessing: {_processed.shape[1]}')


## 4. Model Training

### 4.1 Helper Functions


In [ ]:
def evaluate_with_cv(model, X, y, cv_folds):
    """
    Evaluate a model using stratified k-fold cross-validation.

    Args:
        model: Scikit-learn compatible classifier
        X: Feature DataFrame
        y: Target Series
        cv_folds: Number of CV folds (integer)

    Returns:
        dict with mean metrics ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')
        and 'std' sub-dict with per-metric standard deviations
    """
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []}

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        y_pred_proba = model.predict_proba(X_val)[:, 1]

        metrics['accuracy'].append(accuracy_score(y_val, y_pred))
        metrics['precision'].append(precision_score(y_val, y_pred, pos_label=1))
        metrics['recall'].append(recall_score(y_val, y_pred, pos_label=1))
        metrics['f1'].append(f1_score(y_val, y_pred, pos_label=1))
        metrics['roc_auc'].append(roc_auc_score(y_val == 1, y_pred_proba))

    results = {metric: np.mean(values) for metric, values in metrics.items()}
    results['std'] = {metric: np.std(values) for metric, values in metrics.items()}
    return results


def print_cv_results(results, model_name='Model'):
    """
    Print cross-validation results in a readable format.

    Args:
        results: Output dict from evaluate_with_cv()
        model_name: Label for the model in output headers
    """
    print(f'\n{model_name} - CV Results:')
    print(f"  Accuracy:  {results['accuracy']:.4f} (±{results['std']['accuracy']:.4f})")
    print(f"  F1-Score:  {results['f1']:.4f} (±{results['std']['f1']:.4f})")
    print(f"  Precision: {results['precision']:.4f} (±{results['std']['precision']:.4f})")
    print(f"  Recall:    {results['recall']:.4f} (±{results['std']['recall']:.4f})")
    print(f"  ROC-AUC:   {results['roc_auc']:.4f} (±{results['std']['roc_auc']:.4f})")


In [ ]:
def optimize_xgboost(X, y, random_state, cv_folds):
    """
    Optimize XGBoost hyperparameters with GridSearchCV (Phase 1: broad search).

    Grid is centered around known high-performing parameters (~80% baseline).
    Total combinations: 3^7 = 2,187.

    Args:
        X: Feature DataFrame
        y: Target Series
        random_state: Random seed for reproducibility
        cv_folds: Number of CV folds (integer)

    Returns:
        dict with 'model', 'score', 'params', 'improvement'
    """
    param_grid = {
        'learning_rate': [0.02, 0.03, 0.04],
        'n_estimators': [175, 200, 225],
        'max_depth': [3, 4, 5],
        'min_child_weight': [2, 3, 4],
        'subsample': [0.92, 0.95, 0.98],
        'colsample_bytree': [0.72, 0.75, 0.78],
        'scale_pos_weight': [2.25, 2.5, 2.75]
    }

    xgb = XGBClassifier(
        random_state=random_state,
        eval_metric='logloss',
        n_jobs=-1,
        verbosity=0
    )
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    grid_search = GridSearchCV(
        estimator=xgb, param_grid=param_grid, cv=cv,
        scoring='f1', n_jobs=-1, verbose=1
    )

    print('Optimizing XGBoost (Phase 1)...')
    print(f'  Combinations to try: {3**7}')
    print('  Estimated time: ~15-20 minutes')

    grid_search.fit(X, y)

    print(f'Best F1-Score: {grid_search.best_score_:.4f}')
    print(f'Best parameters: {grid_search.best_params_}')

    return {
        'model': grid_search.best_estimator_,
        'score': grid_search.best_score_,
        'params': grid_search.best_params_,
        'improvement': grid_search.best_score_ > 0.80
    }


### Random Forest (Optional)

`optimize_random_forest()` is defined below but **not called by default**. It is available for comparison experiments.

Usage example:
```python
rf_results = optimize_random_forest(X, y, RANDOM_STATE, CV_FOLDS)
print_cv_results(evaluate_with_cv(rf_results['model'], X, y, CV_FOLDS), 'Random Forest')
```


In [ ]:
def optimize_random_forest(X, y, random_state, cv_folds):
    """
    Optimize RandomForest hyperparameters with GridSearchCV.

    Not called by default — see the markdown cell above for usage.
    Total combinations: 3*3*3*3*1*1 = 81.

    Args:
        X: Feature DataFrame
        y: Target Series
        random_state: Random seed for reproducibility
        cv_folds: Number of CV folds (integer)

    Returns:
        dict with 'model', 'score', 'params'
    """
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt'],
        'class_weight': ['balanced']
    }

    rf = RandomForestClassifier(random_state=random_state)
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    grid_search = GridSearchCV(
        estimator=rf, param_grid=param_grid, cv=cv,
        scoring='f1', n_jobs=-1, verbose=1
    )

    print('Optimizing Random Forest...')
    grid_search.fit(X, y)

    print(f'Best F1-Score: {grid_search.best_score_:.4f}')
    print(f'Best parameters: {grid_search.best_params_}')

    return {
        'model': grid_search.best_estimator_,
        'score': grid_search.best_score_,
        'params': grid_search.best_params_
    }


### 4.2 Execution


In [ ]:
# Preprocess training data and split into features and target
data_processed = preprocess_loan_data(train_df)
X = (
    data_processed.drop(['Loan_Status', 'id'], axis=1)
    if 'id' in data_processed.columns
    else data_processed.drop('Loan_Status', axis=1)
)
y = data_processed['Loan_Status']
print(f'Training features: {X.shape[1]}, Samples: {X.shape[0]}')

# Phase 1: broad hyperparameter search
xgb_results = optimize_xgboost(X, y, RANDOM_STATE, CV_FOLDS)

# Detailed evaluation of Phase 1 best model
phase1_results = evaluate_with_cv(xgb_results['model'], X, y, CV_FOLDS)
print_cv_results(phase1_results, 'XGBoost Phase 1')


In [ ]:
# Phase 2: fine-tuning — micro-optimization around Phase 1 best params
print('\n' + '='*60)
print('PHASE 2: FINE-TUNING')
print('='*60)

base_params = xgb_results['params']
print(f'Base parameters: {base_params}')

# Build fine-grained grid around Phase 1 optimal params
param_grid = {}

base_lr = base_params.get('learning_rate', 0.03)
param_grid['learning_rate'] = [max(0.01, base_lr - 0.005), base_lr, min(0.1, base_lr + 0.005)]

base_n_est = base_params.get('n_estimators', 200)
param_grid['n_estimators'] = [max(50, base_n_est - 25), base_n_est, base_n_est + 25]

base_colsample = base_params.get('colsample_bytree', 0.75)
param_grid['colsample_bytree'] = [
    max(0.5, base_colsample - 0.05), base_colsample, min(1.0, base_colsample + 0.05)
]

base_scale = base_params.get('scale_pos_weight', 2.5)
param_grid['scale_pos_weight'] = [max(1.0, base_scale - 0.25), base_scale, base_scale + 0.25]

# Keep remaining params fixed at Phase 1 best values
param_grid['max_depth'] = [base_params.get('max_depth', 3)]
param_grid['min_child_weight'] = [base_params.get('min_child_weight', 3)]
param_grid['subsample'] = [base_params.get('subsample', 0.95)]

xgb_fine = XGBClassifier(
    random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1, verbosity=0
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fine_grid_search = GridSearchCV(
    estimator=xgb_fine, param_grid=param_grid, cv=cv,
    scoring='f1', n_jobs=-1, verbose=1
)
fine_grid_search.fit(X, y)

fine_tune_results = {
    'model': fine_grid_search.best_estimator_,
    'params': fine_grid_search.best_params_,
    'score': fine_grid_search.best_score_
}
fine_detailed = evaluate_with_cv(fine_tune_results['model'], X, y, CV_FOLDS)
print_cv_results(fine_detailed, 'XGBoost Phase 2 (Fine-tuned)')

# Select the best model between Phase 1 and Phase 2
if fine_detailed['f1'] > phase1_results['f1']:
    print('\nDecision: Using FINE-TUNED model (higher F1)')
    final_best_model = fine_tune_results['model']
    final_best_params = fine_tune_results['params']
    final_best_results = fine_detailed
    model_source = 'Fine-tuned'
else:
    print('\nDecision: Keeping PHASE 1 model (fine-tuning did not improve)')
    final_best_model = xgb_results['model']
    final_best_params = xgb_results['params']
    final_best_results = phase1_results
    model_source = 'Optimized'


## 5. Evaluation


In [ ]:
print_cv_results(final_best_results, f'XGBoost {model_source}')


In [ ]:
# Confusion matrix using cross-validated predictions
y_pred_cv = cross_val_predict(final_best_model, X, y, cv=CV_FOLDS)
ConfusionMatrixDisplay.from_predictions(
    y, y_pred_cv, display_labels=['Rejected', 'Approved']
)
plt.title(f'Confusion Matrix ({CV_FOLDS}-Fold Cross-Validated Predictions)')
plt.tight_layout()
plt.show()


In [ ]:
feature_importance = (
    pd.DataFrame({
        'feature': X.columns,
        'importance': final_best_model.feature_importances_
    })
    .sort_values('importance', ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][::-1], feature_importance['importance'][::-1])
plt.xlabel('Feature Importance')
plt.title('Top 10 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()


## 6. Prediction & Submission

Uses `test_df` loaded in Section 2 — no duplicate file read.


In [ ]:
# Reuse test_df from Section 2
print(f'Test dataset shape: {test_df.shape}')
test_ids = test_df['id'].copy()

# Apply the same preprocessing pipeline used in training
test_processed = preprocess_loan_data(test_df)
X_test = test_processed.copy()
if 'id' in X_test.columns:
    X_test = X_test.drop('id', axis=1)

# Align test columns with training feature set
missing_cols = set(X.columns) - set(X_test.columns)
extra_cols = set(X_test.columns) - set(X.columns)
for col in missing_cols:
    X_test[col] = 0
X_test = X_test.drop(columns=extra_cols)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
print(f'Test features aligned: {X_test.shape[1]}')

# Generate predictions with the best model
test_predictions = final_best_model.predict(X_test)

# Build submission DataFrame
submission = pd.DataFrame({
    'id': test_ids,
    'pred': test_predictions.astype(int)
})

# Save to file using DATA_DIR config constant
submission_filename = os.path.join(DATA_DIR, f'submission_{model_source.lower()}_final.csv')
submission.to_csv(submission_filename, index=False)
print(f'Submission saved: {submission_filename}')
print(submission.head(10).to_string(index=False))


## 7. Interpretability (SHAP)

SHAP (SHapley Additive exPlanations) values measure each feature's contribution to a specific prediction. Positive SHAP values push the prediction toward loan approval; negative values push toward rejection.


In [ ]:
# ANÁLISIS SHAP - INTERPRETABILIDAD DEL MODELO
# Ejecutar DESPUÉS de tener el modelo final optimizado

def analyze_model_interpretability(model, X, y, model_name="XGBoost"):
    """
    Análisis completo de interpretabilidad usando SHAP

    Args:
        model: Modelo entrenado (no se re-entrena)
        X, y: Datos de entrenamiento
        model_name: Nombre para reportes
    """

    try:
        import shap
    except ImportError:
        print("SHAP not installed. Install with: pip install shap")
        return None

    print(f"ANÁLISIS DE INTERPRETABILIDAD - {model_name}")
    print("=" * 60)

    # 1. Crear explainer (no re-entrena el modelo)
    print("Creando SHAP explainer...")
    explainer = shap.TreeExplainer(model)

    # Usar muestra para eficiencia si dataset es grande
    sample_size = min(500, len(X))
    if len(X) > sample_size:
        sample_idx = np.random.choice(len(X), sample_size, replace=False)
        X_sample = X.iloc[sample_idx]
        y_sample = y.iloc[sample_idx]
        print(f"Usando muestra de {sample_size} observaciones para eficiencia")
    else:
        X_sample = X
        y_sample = y
        sample_idx = np.arange(len(X))

    # 2. Calcular SHAP values
    print("Calculando SHAP values...")
    shap_values = explainer.shap_values(X_sample)

    # 3. Feature importance global
    print(f"\nIMPORTANCIA GLOBAL DE FEATURES:")
    print("-" * 40)

    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'mean_abs_shap': np.abs(shap_values).mean(0),
        'mean_shap': np.mean(shap_values, 0)
    }).sort_values('mean_abs_shap', ascending=False)

    print("Top 15 features más importantes:")
    for i, row in feature_importance.head(15).iterrows():
        direction = "→" if row['mean_shap'] > 0 else "←"
        print(f"  {i+1:2d}. {row['feature']:<30} = {row['mean_abs_shap']:.4f} {direction}")

    # 4. Identificar features del preprocessing optimizado en top rankings
    engineered_features = [feat for feat in feature_importance.head(10)['feature']
                          if any(keyword in feat.lower() for keyword in
                                ['missing', 'interaction', 'synergy', 'conservative', 'mismatch', 'stability'])]

    if engineered_features:
        print(f"\nFEATURES DE ENGINEERING EN TOP 10:")
        for feat in engineered_features:
            rank = feature_importance[feature_importance['feature'] == feat].index[0] + 1
            importance = feature_importance[feature_importance['feature'] == feat]['mean_abs_shap'].iloc[0]
            print(f"  #{rank}: {feat} (SHAP: {importance:.4f})")

    # 5. Análisis de errores - False Positives
    y_pred_proba = model.predict_proba(X_sample)[:, 1]

    false_positive_mask = (y_sample == 0) & (y_pred_proba > 0.7)
    fp_indices = np.where(false_positive_mask)[0]

    if len(fp_indices) > 0:
        print(f"\nANÁLISIS DE FALSE POSITIVES:")
        print(f"Casos con alta confianza pero incorrectos: {len(fp_indices)}")
        print("-" * 40)

        # Contribuciones promedio en false positives
        fp_shap = shap_values[fp_indices].mean(0)
        fp_analysis = pd.DataFrame({
            'feature': X.columns,
            'avg_fp_contribution': fp_shap
        }).sort_values('avg_fp_contribution', ascending=False)

        print("Features que más contribuyen a false positives:")
        for i, row in fp_analysis.head(8).iterrows():
            print(f"  {row['feature']:<30} = {row['avg_fp_contribution']:+.4f}")

    # 6. Análisis de errores - False Negatives
    false_negative_mask = (y_sample == 1) & (y_pred_proba < 0.3)
    fn_indices = np.where(false_negative_mask)[0]

    if len(fn_indices) > 0:
        print(f"\nANÁLISIS DE FALSE NEGATIVES:")
        print(f"Casos rechazados incorrectamente: {len(fn_indices)}")
        print("-" * 40)

        fn_shap = shap_values[fn_indices].mean(0)
        fn_analysis = pd.DataFrame({
            'feature': X.columns,
            'avg_fn_contribution': fn_shap
        }).sort_values('avg_fn_contribution', ascending=True)

        print("Features que más contribuyen a false negatives:")
        for i, row in fn_analysis.head(8).iterrows():
            print(f"  {row['feature']:<30} = {row['avg_fn_contribution']:+.4f}")

    # 7. Casos específicos más problemáticos
    if len(fp_indices) > 0:
        print(f"\nCASOS MÁS PROBLEMÁTICOS (False Positives):")
        print("-" * 40)

        # Tomar los 2 peores casos
        worst_fp_idx = fp_indices[np.argsort(y_pred_proba[fp_indices])[-2:]]

        for i, idx in enumerate(worst_fp_idx):
            print(f"\nCaso {i+1} (probabilidad: {y_pred_proba[idx]:.3f}):")

            case_shap = pd.DataFrame({
                'feature': X.columns,
                'shap_value': shap_values[idx],
                'feature_value': X_sample.iloc[idx].values
            }).sort_values('shap_value', ascending=False)

            print("  Top 5 contribuciones positivas:")
            for _, row in case_shap.head(5).iterrows():
                print(f"    {row['feature']:<25}: +{row['shap_value']:.3f} (valor: {row['feature_value']:.3f})")

    # 8. Estadísticas de SHAP
    print(f"\nESTADÍSTICAS SHAP:")
    print("-" * 30)
    print(f"Expected value: {explainer.expected_value:.4f}")
    print(f"SHAP values shape: {shap_values.shape}")
    print(f"Rango de SHAP values: [{shap_values.min():.3f}, {shap_values.max():.3f}]")

    return {
        'explainer': explainer,
        'shap_values': shap_values,
        'feature_importance': feature_importance,
        'X_sample': X_sample,
        'sample_indices': sample_idx
    }


def create_shap_visualizations(shap_results):
    """
    Crear visualizaciones SHAP (ejecutar en celda separada)
    """

    print("CREANDO VISUALIZACIONES SHAP:")
    print("=" * 40)

    explainer = shap_results['explainer']
    shap_values = shap_results['shap_values']
    X_sample = shap_results['X_sample']

    try:
        import matplotlib.pyplot as plt

        # 1. Summary plot
        print("1. Creando summary plot...")
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
        plt.title("SHAP Summary Plot - Feature Importance")
        plt.tight_layout()
        plt.show()

        # 2. Feature importance bar plot
        print("2. Creando bar plot...")
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False, max_display=15)
        plt.title("SHAP Feature Importance")
        plt.tight_layout()
        plt.show()

        # 3. Waterfall para caso específico
        print("3. Creando waterfall plot para caso ejemplo...")
        if hasattr(shap, 'waterfall_plot'):
            shap.waterfall_plot(explainer.expected_value, shap_values[0], X_sample.iloc[0])
        else:
            print("   Waterfall plot no disponible en esta versión de SHAP")

        print("Visualizaciones completadas")

    except Exception as e:
        print(f"Error creando visualizaciones: {e}")
        print("Para crear visualizaciones manualmente:")
        print("  shap.summary_plot(shap_values, X_sample)")
        print("  shap.summary_plot(shap_values, X_sample, plot_type='bar')")


# Función integrada para usar en tu pipeline
def run_interpretability_analysis(model, X, y):
    """
    Run complete SHAP interpretability analysis.

    analyze_model_interpretability retains its original signature
    (model, X, y, model_name='XGBoost') where model_name has a default.

    Args:
        model: Trained classifier (XGBoost or compatible tree model)
        X: Feature DataFrame used for training
        y: Target Series (used for error analysis -- False Positives/Negatives)

    Returns:
        dict with explainer, shap_values, feature_importance, X_sample
    """
    print('STARTING INTERPRETABILITY ANALYSIS')
    print('=' * 60)

    shap_results = analyze_model_interpretability(model, X, y)

    if shap_results:
        print('
Analysis complete.')
        print('To visualize: create_shap_visualizations(shap_results)')

    return shap_results

In [ ]:
shap_results = run_interpretability_analysis(final_best_model, X, y)


In [ ]:
create_shap_visualizations(shap_results)
